# Independently trained random-mask sparse alternative

This notebook contains code selected from the authors’ research notebooks because its analysis is described in the accepted COST manuscript. Outputs and execution counts have been removed. It may require authorized datasets, saved model artifacts, and variables established by preceding pipeline notebooks. See `docs/EXECUTION_ORDER.md`.


# Independently trained random-mask sparse alternative

This notebook documents one experiment reported in the COST manuscript.


In [ ]:
# CELL 1 — SETUP, PATHS, FLAGS
import os, sys, json, math, time, random, warnings, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Not running in Colab (or Drive already mounted):", e)

BASE       = Path(os.environ.get("COST_BASE", "/content/drive/MyDrive/COST_PROJECT"))
MIMIC_BASE = BASE / "mimic-iv-2.1"
HOSP       = MIMIC_BASE / "hosp"
ICU        = MIMIC_BASE / "icu"
CACHE_FILE = MIMIC_BASE / "mimic_cache" / "hf_cohort_processed.npz"
PRED_DIR   = MIMIC_BASE / "saved_predictions"
SAVE_DIR   = MIMIC_BASE / "saved_models"
REV_DIR    = MIMIC_BASE / "revision2_outputs"
CKPT_MAIN  = SAVE_DIR / "dsrnn_v2_best.pt"
ID_FILE    = REV_DIR / "REV6_FINAL_exact_identifiers.csv"
EICU_PRED  = REV_DIR / "external_eicu" / "eicu_external_predictions.npz"
OUT        = REV_DIR / "final_round_reviewer"
OUT.mkdir(parents=True, exist_ok=True)

# ---------------- FLAGS ----------------
RUN_COHORT_AUDIT         = True   # A1/A2: reads admissions.csv + icustays.csv (small, read-only)
RUN_CHECKPOINT_INFERENCE = True   # A4-A6: rebuild preprocessing + inference with frozen checkpoints
RUN_GATING_SEED_RECAL    = True   # A5: also recalibrate gating ON/OFF seed checkpoints (if *_best.pt exist)
RUN_EICU                 = True   # A8: only runs if EICU_PRED exists
RUN_OPTIONAL_TRAINING    = False  # A9: NEW TRAINING (≈1–2 h GPU). Keep False unless you decide to add it.
N_BOOT   = 2000
SEED     = 42
TRAIN_YEARS, VAL_YEARS, TEST_YEARS = range(2008, 2016), range(2016, 2018), range(2018, 2020)

VITAL_SIGNS = ['heart_rate', 'sbp', 'dbp', 'map', 'resp_rate', 'spo2', 'temperature']
LAB_VALUES  = ['creatinine', 'bnp', 'lactate', 'troponin', 'sodium', 'potassium', 'chloride',
               'bicarbonate', 'bun', 'glucose', 'hemoglobin', 'wbc', 'platelets', 'inr', 'pt',
               'ptt', 'ph', 'pao2', 'paco2', 'fio2', 'albumin', 'bilirubin_total', 'alt', 'ast']
STATIC_FEATURES = ['age', 'gender_male', 'admission_emergency', 'admission_urgent', 'elixhauser_score',
                   'hypertension', 'diabetes', 'ckd', 'copd', 'af', 'prior_hf']
ALL_TEMPORAL_FEATURES = VITAL_SIGNS + LAB_VALUES
N_FEATURES, N_STATIC = len(ALL_TEMPORAL_FEATURES), len(STATIC_FEATURES)
N_INPUT = N_FEATURES * 3

REPORT = {}      # key numbers for the response letter
FLAGS  = []      # audit notes to verify against the final pipeline

# Saved test predictions to evaluate: display name -> (file in PRED_DIR, key). Edit if your final files differ.
PRED_FILES = {
    "Logistic Regression": ("baseline_lr_predictions.npz", "probs"),
    "Random Forest":       ("baseline_rf_predictions.npz", "probs"),
    "Standard LSTM":       ("baseline_lstm_predictions.npz", "probs"),
    "Attention LSTM":      ("baseline_attn_lstm_predictions.npz", "probs"),
    "TCN":                 ("baseline_tcn_predictions.npz", "probs"),
    "COST (Initial)":      ("cost_before_predictions.npz", "probs"),
    "COST (Improved) single":   ("cost_improved_predictions.npz", "probs_single"),
    "COST (Improved) ensemble": ("cost_improved_predictions.npz", "probs_ensemble"),
}

GATING_SEEDS = [42, 43, 44]   # Revision II gating ON/OFF artefacts in REV_DIR (skipped if absent)

random.seed(SEED); np.random.seed(SEED)
for p in [CACHE_FILE, PRED_DIR, ID_FILE, CKPT_MAIN]:
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")
print("Output folder:", OUT)

In [ ]:
# CELL 2 — UTILITIES: metrics, DeLong, (cluster) bootstrap, calibration, recalibration, plots
def show(df, n=60):
    try:
        display(df.head(n))          # noqa: F821  (Colab/Jupyter)
    except NameError:
        print(df.head(n).to_string())

def save_table(df, name):
    df.to_csv(OUT / f"{name}.csv", index=False)
    TABLES[name] = df
    return df
TABLES = {}

def sens_at_spec(y, p, target_spec=0.90):
    # identical definition to the manuscript code
    fpr, tpr, _ = roc_curve(y, p)
    idx = np.argmin(np.abs((1 - fpr) - target_spec))
    return float(tpr[idx])

def _clip(p): return np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
def logit(p): p = _clip(p); return np.log(p / (1 - p))
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def fit_logistic(X, y, offset=None, iters=100):
    # Newton-Raphson logistic regression; X must include an intercept column if wanted
    X = np.asarray(X, float); y = np.asarray(y, float)
    off = np.zeros(len(y)) if offset is None else np.asarray(offset, float)
    b = np.zeros(X.shape[1])
    for _ in range(iters):
        mu = sigmoid(X @ b + off)
        W = mu * (1 - mu)
        H = X.T @ (X * W[:, None]) + 1e-9 * np.eye(X.shape[1])
        step = np.linalg.solve(H, X.T @ (y - mu))
        b += step
        if np.max(np.abs(step)) < 1e-8:
            break
    return b

def ece_score(y, p, n_bins=10):
    y = np.asarray(y, float); p = np.asarray(p, float)
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)

def calib_stats(y, p):
    y = np.asarray(y, float); p = np.asarray(p, float); z = logit(p)
    prev = y.mean()
    slope = fit_logistic(np.c_[np.ones_like(z), z], y)[1]
    citl  = fit_logistic(np.ones((len(y), 1)), y, offset=z)[0]   # calibration-in-the-large
    brier = brier_score_loss(y, p)
    return {"mean_pred": p.mean(), "observed": prev, "citl_intercept": citl, "calib_slope": slope,
            "ECE": ece_score(y, p), "Brier": brier, "Brier_ref": prev * (1 - prev),
            "scaled_Brier": 1 - brier / (prev * (1 - prev))}

def full_metrics(y, p):
    y = np.asarray(y).astype(int); p = np.asarray(p, float)
    d = {"N": len(y), "deaths": int(y.sum()),
         "AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p),
         "Sens@90Spec": sens_at_spec(y, p)}
    d.update(calib_stats(y, p))
    return d

# ---- DeLong (fast, Sun & Xu 2014) ----
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    out = np.empty(N); out[J] = T + 1
    return out

def delong_paired(y, p1, p2):
    y = np.asarray(y).astype(int)
    order = np.argsort(-y, kind="mergesort")      # positives first
    m = int(y.sum()); n = len(y) - m
    P = np.vstack([np.asarray(p1, float), np.asarray(p2, float)])[:, order]
    tx = np.vstack([_midrank(r[:m]) for r in P]); ty = np.vstack([_midrank(r[m:]) for r in P])
    tz = np.vstack([_midrank(r) for r in P])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    l = np.array([1, -1]); var = float(l @ cov @ l)
    zstat = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else np.nan
    pval = 2 * stats.norm.sf(abs(zstat)) if var > 0 else np.nan
    return float(aucs[0]), float(aucs[1]), float(zstat), float(pval)

# ---- Bootstrap (optionally clustered by subject_id) ----
def _boot_indices(n, clusters, rng):
    if clusters is None:
        return rng.integers(0, n, n)
    uniq = np.unique(clusters)
    pick = rng.choice(uniq, size=len(uniq), replace=True)
    pos = {c: np.flatnonzero(clusters == c) for c in uniq}
    return np.concatenate([pos[c] for c in pick])

def boot_ci(y, p, fn=roc_auc_score, n_boot=N_BOOT, clusters=None, seed=SEED):
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); vals = []
    for _ in range(n_boot):
        idx = _boot_indices(len(y), clusters, rng)
        if y[idx].min() == y[idx].max():
            continue
        vals.append(fn(y[idx], p[idx]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, p1, p2, fn=roc_auc_score, n_boot=N_BOOT, clusters=None, seed=SEED):
    rng = np.random.default_rng(seed); y = np.asarray(y); d = []
    for _ in range(n_boot):
        idx = _boot_indices(len(y), clusters, rng)
        if y[idx].min() == y[idx].max():
            continue
        d.append(fn(y[idx], p1[idx]) - fn(y[idx], p2[idx]))
    d = np.asarray(d)
    p_two = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return float(np.mean(d)), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float(min(p_two, 1.0))

# ---- Recalibration ----
def prior_shift(p, target_prev, train_prev=0.5):
    # analytic correction: model trained with a class-balanced sampler (effective prior 0.5)
    return sigmoid(logit(p) + np.log(target_prev / (1 - target_prev)) - np.log(train_prev / (1 - train_prev)))

class Platt:
    def fit(self, p, y):
        z = logit(p); self.b = fit_logistic(np.c_[np.ones_like(z), z], y); return self
    def predict(self, p):
        z = logit(p); return sigmoid(self.b[0] + self.b[1] * z)

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(np.asarray(p), np.asarray(y)); return self
    def predict(self, p):
        return np.clip(self.m.predict(np.asarray(p)), 1e-6, 1 - 1e-6)

class InterceptOnly:
    def fit(self, p, y):
        self.a = fit_logistic(np.ones((len(y), 1)), y, offset=logit(p))[0]; return self
    def predict(self, p):
        return sigmoid(logit(p) + self.a)

def crossfit_recal(y, p, maker, n_splits=5, n_repeats=20, seed=SEED):
    # out-of-fold recalibration on the evaluation set itself (supplementary; no information leak per fold)
    y = np.asarray(y).astype(int); p = np.asarray(p, float); rows = []; oof0 = None
    for r in range(n_repeats):
        oof = np.zeros_like(p)
        for tr, te in StratifiedKFold(n_splits, shuffle=True, random_state=seed + r).split(p, y):
            oof[te] = maker().fit(p[tr], y[tr]).predict(p[te])
        rows.append(calib_stats(y, oof))
        if oof0 is None:
            oof0 = oof
    return pd.DataFrame(rows).mean().to_dict(), oof0

def reliability_plot(curves, y, title, fname, n_bins=10):
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    for name, p in curves.items():
        p = np.asarray(p); bins = np.linspace(0, 1, n_bins + 1)
        idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
        xs, ys = [], []
        for b in range(n_bins):
            m = idx == b
            if m.sum() >= 5:
                xs.append(p[m].mean()); ys.append(np.asarray(y)[m].mean())
        ax.plot(xs, ys, "o-", ms=4, label=f"{name} (Brier={brier_score_loss(y, p):.3f})")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed mortality")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title(title, fontsize=10); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / fname, dpi=300); plt.show(); plt.close(fig)

print("Utilities ready")

In [ ]:
# CELL 3 — LOAD CACHE, IDENTIFIERS, SAVED PREDICTIONS + ALIGNMENT CHECKS
cache = np.load(CACHE_FILE, allow_pickle=True)
X_temporal_raw = cache["X_temporal"].astype(np.float32)
X_static_raw   = cache["X_static"].astype(np.float32)
labels_all     = cache["labels"].astype(int)
years_all      = cache["years"].astype(int)
N_ALL = len(labels_all)
print("Cache:", X_temporal_raw.shape, X_static_raw.shape, "deaths", labels_all.sum())

HAS_IDS = ID_FILE.exists()
if HAS_IDS:
    ids = pd.read_csv(ID_FILE)
    HAS_IDS = (len(ids) == N_ALL and np.array_equal(ids["label"].to_numpy().astype(int), labels_all)
               and np.array_equal(ids["real_admit_year"].to_numpy().astype(int), years_all))
    print("Identifier file aligned with cache order:", "OK" if HAS_IDS else "NO — subject-level analyses will be skipped")
if not HAS_IDS:
    print("No aligned identifier file: building placeholder IDs (A0/A1 subject & 48h analyses skipped).")
    ids = pd.DataFrame({"subject_id": np.arange(N_ALL), "hadm_id": np.arange(N_ALL), "stay_id": np.arange(N_ALL),
                        "real_admit_year": years_all, "label": labels_all})
    RUN_COHORT_AUDIT = False

split_of = np.where(np.isin(years_all, list(TRAIN_YEARS)), "train",
           np.where(np.isin(years_all, list(VAL_YEARS)), "val",
           np.where(np.isin(years_all, list(TEST_YEARS)), "test", "none")))
ids["split"] = split_of
test_mask, val_mask, train_mask = split_of == "test", split_of == "val", split_of == "train"
ids_test = ids[test_mask].reset_index(drop=True)
y_test = ids_test["label"].to_numpy().astype(int)
print({s: int((split_of == s).sum()) for s in ["train", "val", "test"]})

PREDS = {}
for name, (fn, key) in PRED_FILES.items():
    f = PRED_DIR / fn
    if not f.exists():
        print("missing:", fn); continue
    d = np.load(f, allow_pickle=True)
    assert np.array_equal(d["test_labels"].astype(int), y_test), f"{fn}: test label/order mismatch"
    PREDS[name] = d[key].astype(np.float64)

GATING = {}   # (arm, seed) -> dict(single, ensemble)
for arm in ["ON", "OFF"]:
    for s in GATING_SEEDS:
        f = REV_DIR / f"gating_{arm}_seed{s}_predictions.npz"
        if f.exists():
            d = np.load(f, allow_pickle=True)
            assert np.array_equal(d["test_labels"].astype(int), y_test)
            GATING[(arm, s)] = {"single": d["probs_single"].astype(float), "ensemble": d["probs_ensemble"].astype(float)}
print("Loaded predictions:", list(PREDS), "| gating artefacts:", sorted(GATING))
show(pd.DataFrame([{"Model": k, "AUROC": roc_auc_score(y_test, v), "AUPRC": average_precision_score(y_test, v),
                    "Brier": brier_score_loss(y_test, v), "min p": v.min(), "max p": v.max(), "mean p": v.mean()}
                   for k, v in PREDS.items()]).round(4))

In [ ]:
# CELL 8 — A4: EXACT PREPROCESSING V2 (vectorised, verified) + DATA AUDIT
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

def add_missingness_features_loop(X):          # original implementation (used only for verification)
    n, seq, feats = X.shape
    mask = (~np.isnan(X)).astype(np.float32)
    delta = np.zeros_like(X, dtype=np.float32)
    for i in range(n):
        for k in range(feats):
            last_obs = 0
            for t in range(seq):
                if not np.isnan(X[i, t, k]):
                    last_obs = t; delta[i, t, k] = 0.0
                else:
                    delta[i, t, k] = float(t - last_obs)
    delta = delta / (seq + 1e-8)
    values = X.copy()
    for i in range(n):
        for k in range(len(VITAL_SIGNS)):
            last = 0.0
            for t in range(seq):
                if not np.isnan(values[i, t, k]):
                    last = values[i, t, k]
                else:
                    values[i, t, k] = last
    values = np.nan_to_num(values, nan=0.0)
    return np.concatenate([values, mask, delta], axis=-1).astype(np.float32)

def add_missingness_features(X):               # vectorised, identical output
    n, seq, feats = X.shape
    obs = ~np.isnan(X)
    mask = obs.astype(np.float32)
    t = np.arange(seq, dtype=np.int64)[None, :, None]
    last_obs = np.maximum.accumulate(np.where(obs, t, 0), axis=1)
    delta = np.where(obs, 0.0, (t - last_obs)).astype(np.float32) / np.float32(seq + 1e-8)
    values = X.copy()
    nv = len(VITAL_SIGNS)
    v = values[:, :, :nv]; ov = obs[:, :, :nv]
    idx = np.maximum.accumulate(np.where(ov, t[:, :, :1].repeat(nv, axis=2), -1), axis=1)
    filled = np.take_along_axis(np.nan_to_num(v, nan=0.0), np.clip(idx, 0, None), axis=1)
    values[:, :, :nv] = np.where(idx >= 0, filled, 0.0)
    values = np.nan_to_num(values, nan=0.0)
    return np.concatenate([values, mask, delta], axis=-1).astype(np.float32)

if RUN_CHECKPOINT_INFERENCE:
    # ---- verify vectorised == loop on a subset (after the same lab handling as training) ----
    sub = X_temporal_raw[:300].copy()
    sub[:, :, len(VITAL_SIGNS):] = np.nan_to_num(sub[:, :, len(VITAL_SIGNS):], nan=0.0, posinf=0.0, neginf=0.0)
    assert np.allclose(add_missingness_features(sub), add_missingness_features_loop(sub), atol=1e-7)
    print("Vectorised missingness features identical to original loop: OK")

    def preprocess_v2(X_temporal, X_static, labels, years):
        n_vital = len(VITAL_SIGNS); splits_raw = {}
        for name, yrs in [("train", TRAIN_YEARS), ("val", VAL_YEARS), ("test", TEST_YEARS)]:
            m = np.isin(years, list(yrs))
            splits_raw[name] = {"X_temp": X_temporal[m].copy(), "X_stat": X_static[m].copy(), "y": labels[m].copy()}
        audit = {"lab_nan_fraction_raw_train": float(np.isnan(splits_raw["train"]["X_temp"][:, :, n_vital:]).mean())}
        imp = IterativeImputer(max_iter=5, random_state=SEED, n_nearest_features=10)
        tr = splits_raw["train"]["X_temp"]; n_tr, seq, n_feat = tr.shape
        lab_train = tr[:, :, n_vital:].reshape(n_tr * seq, n_feat - n_vital)
        lab_train = np.nan_to_num(lab_train, nan=0.0, posinf=0.0, neginf=0.0)   # <- as in original code (before MICE)
        audit["lab_nan_fraction_entering_MICE"] = float(np.isnan(lab_train).mean())
        imp.fit(lab_train)
        for name in splits_raw:
            arr = splits_raw[name]["X_temp"]; n_sp = arr.shape[0]
            lab_d = np.nan_to_num(arr[:, :, n_vital:].reshape(n_sp * seq, n_feat - n_vital), nan=0.0, posinf=0.0, neginf=0.0)
            out = imp.transform(lab_d)
            if name == "train":
                audit["MICE_max_abs_change_train"] = float(np.abs(out - lab_d).max())
            arr[:, :, n_vital:] = out.reshape(n_sp, seq, n_feat - n_vital)
        splits = {}
        for name in splits_raw:
            splits[name] = {"X_temp": add_missingness_features(splits_raw[name]["X_temp"]),
                            "X_stat": splits_raw[name]["X_stat"].copy(), "y": splits_raw[name]["y"].copy()}
        tr_flat = splits["train"]["X_temp"][:, :, :N_FEATURES].reshape(-1, N_FEATURES)
        feat_mean = tr_flat.mean(axis=0); feat_std = tr_flat.std(axis=0) + 1e-8
        stat_mean = splits["train"]["X_stat"].mean(axis=0); stat_std = splits["train"]["X_stat"].std(axis=0) + 1e-8
        for name in splits:
            arr = splits[name]["X_temp"].copy()
            arr[:, :, :N_FEATURES] = (arr[:, :, :N_FEATURES] - feat_mean) / feat_std
            splits[name]["X_temp"] = arr
            splits[name]["X_stat"] = (splits[name]["X_stat"] - stat_mean) / stat_std
        norm = {"feat_mean": feat_mean, "feat_std": feat_std, "stat_mean": stat_mean, "stat_std": stat_std}
        return splits, norm, audit

    t0 = time.time()
    SPLITS, NORM, PREP_AUDIT = preprocess_v2(X_temporal_raw.copy(), X_static_raw.copy(), labels_all, years_all)
    print(f"preprocess_v2 rebuilt in {(time.time()-t0)/60:.1f} min")
    assert np.array_equal(SPLITS["test"]["y"].astype(int), y_test)

    tr = SPLITS["train"]["X_temp"]; nv = len(VITAL_SIGNS)
    lab_mask = tr[:, :, N_FEATURES + nv: 2 * N_FEATURES]; lab_delta = tr[:, :, 2 * N_FEATURES + nv:]
    vit_mask = tr[:, :, N_FEATURES: N_FEATURES + nv]
    temp = X_temporal_raw[:, :, VITAL_SIGNS.index("temperature")]; temp = temp[~np.isnan(temp)]
    PREP_AUDIT.update({
        "lab_mask_unique_values": np.unique(lab_mask).tolist(),
        "lab_delta_all_zero": bool(np.all(lab_delta == 0)),
        "vital_mask_observed_fraction": float(vit_mask.mean()),
        "temperature_values_below_45 (Celsius-like) %": float(100 * (temp < 45).mean()),
        "temperature_values_above_80 (Fahrenheit-like) %": float(100 * (temp > 80).mean()),
    })
    print(json.dumps(PREP_AUDIT, indent=2))
    pd.Series(PREP_AUDIT, dtype=object).to_csv(OUT / "A4_preprocessing_audit.csv")
    if PREP_AUDIT["lab_nan_fraction_entering_MICE"] == 0.0 and PREP_AUDIT["lab_delta_all_zero"]:
        FLAGS.append(
            "[CHECK] In the preprocess_v2 version reproduced here (validated against the checkpoint in A5), lab NaNs are zero-filled "
            f"before IterativeImputer (max MICE change {PREP_AUDIT['MICE_max_abs_change_train']:.2e}); lab masks are constant 1 and lab "
            "delta-t constant 0, so informative missingness/delta-t comes from the 7 vital signs. Verify against Sec. III-D/E, "
            "Table VIII and Table XII.")
    if PREP_AUDIT["temperature_values_below_45 (Celsius-like) %"] > 1 and PREP_AUDIT["temperature_values_above_80 (Fahrenheit-like) %"] > 1:
        FLAGS.append("[CHECK] Temperature channel mixes °F (itemid 223761) and °C (223762) values; mention as a limitation or harmonise in future work.")
    for f in FLAGS[-2:]: print(f)

In [ ]:
# CELL 9 — MODEL (state_dict-compatible with all COST Improved checkpoints) + INFERENCE HELPERS
if RUN_CHECKPOINT_INFERENCE:
    import torch, torch.nn as nn, torch.nn.functional as F
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(SEED)
    print("Device:", DEVICE)

    class SinusoidalPE(nn.Module):
        def __init__(self, d_model, max_len=256, dropout=0.1):
            super().__init__(); self.dropout = nn.Dropout(dropout)
            pe = torch.zeros(max_len, d_model); pos = torch.arange(max_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pe", pe.unsqueeze(0))
        def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1)])

    class StochasticDepth(nn.Module):
        def __init__(self, p=0.1): super().__init__(); self.p = p
        def forward(self, x, residual):
            if not self.training or self.p == 0: return x + residual
            keep = torch.bernoulli(torch.tensor(1 - self.p)).item(); return x + residual * keep

    class ClinicalSalienceV2(nn.Module):
        def __init__(self, input_dim, hidden_dim, n_features):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, hidden_dim * 2); self.norm = nn.LayerNorm(hidden_dim)
            self.fc2 = nn.Linear(hidden_dim, n_features); self.tau = nn.Parameter(torch.tensor(0.5))
        def forward(self, x_t, h_prev):
            h = self.fc1(torch.cat([x_t, h_prev], dim=-1)); h, gate = h.chunk(2, dim=-1)
            h = self.norm(h * torch.sigmoid(gate)); s_t = torch.sigmoid(self.fc2(h))
            m_hard = (s_t > self.tau).float(); return m_hard - s_t.detach() + s_t, s_t

    class TemporalMultiHeadAttn(nn.Module):
        def __init__(self, d_model, n_heads, dropout=0.1):
            super().__init__()
            self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(d_model); self.sdrop = StochasticDepth(p=0.1)
        def forward(self, x):
            out, w = self.attn(x, x, x); return self.norm(self.sdrop(x, out)), w

    class StaticTemporalFusion(nn.Module):
        def __init__(self, d_temporal, n_static, d_out, n_heads=4, dropout=0.1):
            super().__init__()
            self.static_proj = nn.Linear(n_static, d_temporal)
            self.cross_attn = nn.MultiheadAttention(d_temporal, n_heads, dropout=dropout, batch_first=True)
            self.norm = nn.LayerNorm(d_temporal)
            self.ffn = nn.Sequential(nn.Linear(d_temporal * 2, d_temporal), nn.GELU(), nn.Dropout(dropout),
                                     nn.Linear(d_temporal, d_out))
        def forward(self, s, h):
            q = self.static_proj(s).unsqueeze(1); ctx, w = self.cross_attn(q, h, h)
            ctx = self.norm(ctx.squeeze(1)); return self.ffn(torch.cat([ctx, h.mean(dim=1)], dim=-1)), w.squeeze(1)

    class COSTImproved(nn.Module):
        # gate_mode: 'learned' (paper, gating ON) | 'off' (bypass, gating OFF) |
        #            'all_zero' (m=0 → 0.1·z) | 'all_one' (m=1 → z) | 'random' (Bernoulli(rho)) | 'constant' ((0.1+0.9·rho)·z)
        def __init__(self, n_input=N_INPUT, n_static=N_STATIC, hidden_size=256, n_lstm_layers=2, n_heads=4,
                     attention_dim=128, salience_hidden=128, dropout=0.3, lambda_sparse=5e-4, gate_mode="learned", rho=0.0099):
            super().__init__()
            self.hidden_size, self.lambda_sparse, self.gate_mode, self.rho = hidden_size, lambda_sparse, gate_mode, rho
            self.input_proj = nn.Sequential(nn.Linear(n_input, hidden_size), nn.LayerNorm(hidden_size), nn.GELU(), nn.Dropout(dropout))
            self.pos_enc = SinusoidalPE(hidden_size, dropout=dropout * 0.5)
            self.salience = ClinicalSalienceV2(hidden_size * 2, salience_hidden, hidden_size)
            self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers=n_lstm_layers, batch_first=True, bidirectional=True,
                                dropout=dropout if n_lstm_layers > 1 else 0)
            self.bidir_proj = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.LayerNorm(hidden_size), nn.GELU())
            self.temporal_attn = TemporalMultiHeadAttn(hidden_size, n_heads, dropout)
            self.fusion = StaticTemporalFusion(hidden_size, n_static, attention_dim, n_heads, dropout)
            self.classifier = nn.Sequential(nn.LayerNorm(attention_dim), nn.Dropout(dropout),
                                            nn.Linear(attention_dim, attention_dim // 2), nn.GELU(),
                                            nn.Dropout(dropout * 0.5), nn.Linear(attention_dim // 2, 1))
            self._init_weights()
        def _init_weights(self):
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None: nn.init.zeros_(m.bias)
                elif isinstance(m, nn.LSTM):
                    for n_, p_ in m.named_parameters():
                        if "weight" in n_: nn.init.orthogonal_(p_)
                        elif "bias" in n_: nn.init.zeros_(p_)
        def forward(self, x_temporal, x_static, return_masks=False):
            b, seq, _ = x_temporal.shape
            z = self.pos_enc(self.input_proj(x_temporal))
            c = torch.zeros(b, self.hidden_size, device=x_temporal.device)
            gated, masks = [], []; sp = torch.tensor(0.0, device=x_temporal.device)
            for t in range(seq):
                z_t = z[:, t, :]
                if self.gate_mode == "off":
                    zg = z_t
                else:
                    if self.gate_mode == "learned":
                        m_t, _ = self.salience(z_t, c); sp = sp + m_t.norm(p=1)
                    elif self.gate_mode == "all_zero":
                        m_t = torch.zeros_like(z_t)
                    elif self.gate_mode == "all_one":
                        m_t = torch.ones_like(z_t)
                    elif self.gate_mode == "random":
                        m_t = (torch.rand_like(z_t) < self.rho).float()
                    elif self.gate_mode == "constant":
                        m_t = torch.full_like(z_t, self.rho)
                    else:
                        raise ValueError(self.gate_mode)
                    zg = m_t * z_t + (1 - m_t) * z_t * 0.1
                    if return_masks: masks.append(m_t.detach().unsqueeze(1))
                gated.append(zg.unsqueeze(1))
                c = c * 0.9 + zg.detach() * 0.1
            sp = self.lambda_sparse * sp / (seq * b) if self.gate_mode == "learned" else torch.tensor(0.0, device=x_temporal.device)
            lo, _ = self.lstm(torch.cat(gated, dim=1)); lo = self.bidir_proj(lo)
            H, _ = self.temporal_attn(lo); zz, _ = self.fusion(x_static, H)
            logits = self.classifier(zz).squeeze(-1)
            if return_masks:
                return logits, sp, (torch.cat(masks, dim=1) if masks else None)
            return logits, sp

    def load_state(path):
        obj = torch.load(path, map_location="cpu", weights_only=False)
        return obj["model_state_dict"] if isinstance(obj, dict) and "model_state_dict" in obj else obj

    def build_model(state, gate_mode="learned", rho=0.0099):
        m = COSTImproved(gate_mode=gate_mode, rho=rho).to(DEVICE)
        m.load_state_dict(state, strict=True); m.eval(); return m

    @torch.no_grad()
    def predict(model, split, batch=256, return_masks=False, seed=None):
        if seed is not None:
            torch.manual_seed(seed)
        X, S = SPLITS[split]["X_temp"], SPLITS[split]["X_stat"]; probs, masks = [], []
        for i in range(0, len(X), batch):
            xb = torch.from_numpy(X[i:i + batch]).float().to(DEVICE); sb = torch.from_numpy(S[i:i + batch]).float().to(DEVICE)
            if return_masks:
                lg, _, mk = model(xb, sb, return_masks=True); masks.append(mk.cpu().numpy().astype(np.uint8))
            else:
                lg, _ = model(xb, sb)
            probs.append(torch.sigmoid(lg).float().cpu().numpy())
        p = np.concatenate(probs).astype(np.float64)
        return (p, np.concatenate(masks)) if return_masks else p
    print("Model + inference helpers ready")

## Experiment 1 — Independently trained sparse control (matched training protocol)

*Training from random initialization* pada `COSTImproved(gate_mode="random")` menggunakan pipeline v2, seed [42,43,44], epoch/budget dan setup loss yang sama dengan implementasi reported trained sparse-control experiment. **Random mask here is resampled during training**, with expected density ρ from final gate statistics; it is *not* a fixed-mask baseline. Hasilnya dibandingkan dengan learned-gating ON dan OFF yang telah dilatih dalam notebook `_FINAL_SelfContained` (tanpa training ulang kedua arm itu). Laporkan definisi random mask dengan tepat.

Guardrails: density diambil dari `A6_gate_statistics.csv` jika ada, kalau tidak gunakan 0.0099 sesuai hasil 99.01%; validasi per seed dan cross-artefact test label alignment. Tabel hasil final tidak ditulis sampai seluruh tiga seed valid.

In [ ]:
# R2 TARGET SETTINGS + PRE-FLIGHT — explicitly launch three independent training runs
import random, time, copy
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

TARGET_OUT = REV_DIR / 'reviewer2_targeted_completion'
TARGET_OUT.mkdir(parents=True, exist_ok=True)
RUN_TRAINED_SPARSE_CONTROL = True  # This notebook is designed to RUN, not merely describe an experiment.
SPARSE_SEEDS = [42,43,44]
RHO = 0.0099  # matches ~99.01% binary latent sparsity in the final paper
stats_file = OUT / 'A6_gate_statistics.csv'
if stats_file.exists():
    _d = pd.read_csv(stats_file)
    _v = _d.loc[_d['Statistic'] == 'retention_rate (fraction of m_t = 1)', 'Value']
    if len(_v): RHO = float(_v.iloc[0])
assert 0 < RHO < 1, RHO
assert torch.cuda.is_available(), 'Training requested: enable GPU in Colab (Runtime > Change runtime type).'
assert CKPT_MAIN.exists(), f'Main checkpoint missing: {CKPT_MAIN}'
_check = torch.load(CKPT_MAIN, map_location='cpu', weights_only=False)
assert isinstance(_check,dict) and 'config' in _check, 'Training configuration missing in checkpoint.'
C = dict(_check['config'])
required=['batch_size','lr','weight_decay','max_epochs','warmup_pct','focal_alpha','focal_gamma',
          'label_smoothing','rdrop_alpha','grad_clip','patience','n_snapshots']
assert all(k in C for k in required), 'Missing keys: '+str([k for k in required if k not in C])
print('Preflight OK | expected mask retention',RHO,'| checkpoint training config:',C)


In [ ]:
# Matched protocol — derived from original reviewer A9, with deterministic evaluation and saved provenance
class CostData(Dataset):
    def __init__(self, d):
        self.x=torch.as_tensor(d['X_temp'],dtype=torch.float32)
        self.s=torch.as_tensor(d['X_stat'],dtype=torch.float32)
        self.y=torch.as_tensor(d['y'],dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.s[i],self.y[i]

class FocalLossWithSmoothing(nn.Module):
    def __init__(self,alpha=.75,gamma=2.,smoothing=.05):
        super().__init__();self.alpha=alpha;self.gamma=gamma;self.smoothing=smoothing
    def forward(self,logits,y):
        ys=y*(1-self.smoothing)+.5*self.smoothing
        bce=F.binary_cross_entropy_with_logits(logits,ys,reduction='none')
        prob=torch.sigmoid(logits);pt=prob*y+(1-prob)*(1-y)
        alpha_t=self.alpha*y+(1-self.alpha)*(1-y)
        return (alpha_t*(1-pt)**self.gamma*bce).mean()

def rdrop_kl(l1,l2):
    p1=torch.sigmoid(l1).clamp(1e-8,1-1e-8);p2=torch.sigmoid(l2).clamp(1e-8,1-1e-8)
    return .5*(p1*(p1.log()-p2.log())+p2*(p2.log()-p1.log())).mean()

def loaders_for_seed(seed):
    ls={}
    for nm,d in SPLITS.items():
        ds=CostData(d)
        if nm=='train':
            y=d['y'].astype(int);counts=np.bincount(y,minlength=2)
            weights=1./np.maximum(counts[y],1)
            rng=torch.Generator().manual_seed(seed)
            sampler=WeightedRandomSampler(weights,len(y),replacement=True,generator=rng)
            ls[nm]=DataLoader(ds,batch_size=int(C['batch_size']),sampler=sampler,num_workers=0)
        else: ls[nm]=DataLoader(ds,batch_size=int(C['batch_size']),shuffle=False,num_workers=0)
    return ls

@torch.no_grad()
def eval_sparse(model,loader,seed=42):
    model.eval();torch.manual_seed(seed)
    ys=[];ps=[]
    for xt,xs,y in loader:
        logits,_=model(xt.to(DEVICE),xs.to(DEVICE))
        ys.append(y.numpy());ps.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.concatenate(ys).astype(int),np.concatenate(ps)

def train_sparse_control(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    L=loaders_for_seed(seed)
    model=COSTImproved(gate_mode='random',rho=RHO).to(DEVICE)
    opt=AdamW(model.parameters(),lr=float(C['lr']),weight_decay=float(C['weight_decay']),betas=(.9,.999))
    sched=OneCycleLR(opt,max_lr=float(C['lr']),total_steps=int(C['max_epochs'])*len(L['train']),
                     pct_start=float(C['warmup_pct']),anneal_strategy='cos')
    fl=FocalLossWithSmoothing(C['focal_alpha'],C['focal_gamma'],C['label_smoothing'])
    # Keep original R-drop and mixed precision protocol; torch AMP supported on CUDA
    scaler=torch.amp.GradScaler('cuda')
    snaps=[];best=-np.inf;pat=0;hist=[]
    for ep in range(1,int(C['max_epochs'])+1):
        model.train();losses=[]
        for xt,xs,y in L['train']:
            xt,xs,y=xt.to(DEVICE),xs.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                a,sa=model(xt,xs);b,sb=model(xt,xs)
                loss=.5*(fl(a,y)+fl(b,y))+float(C['rdrop_alpha'])*rdrop_kl(a,b)+.5*(sa+sb)
            scaler.scale(loss).backward();scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),float(C['grad_clip']))
            scaler.step(opt);scaler.update();sched.step();losses.append(float(loss.detach().cpu()))
        # Seed evaluation to avoid random-mask RNG noise shifting between epochs
        yv,pv=eval_sparse(model,L['val'],seed=seed+10000)
        val=roc_auc_score(yv,pv)
        snaps.append((val,copy.deepcopy({k:v.detach().cpu() for k,v in model.state_dict().items()})))
        snaps.sort(key=lambda z:z[0],reverse=True);snaps=snaps[:int(C['n_snapshots'])]
        best,pat=(val,0) if val>best else (best,pat+1)
        hist.append({'seed':seed,'epoch':ep,'loss':np.mean(losses),'val_AUROC':val})
        if ep==1 or ep%5==0: print(f'seed={seed} epoch={ep} valAUROC={val:.4f} best={best:.4f}')
        if pat>=int(C['patience']):break
    assert len(snaps)>0
    model.load_state_dict(snaps[0][1])
    yt,ps=eval_sparse(model,L['test'],seed=seed+20000)
    pe=[]
    for _,st in snaps:
        model.load_state_dict(st)
        y_,p_=eval_sparse(model,L['test'],seed=seed+20000)
        assert np.array_equal(y_,yt)
        pe.append(p_)
    pens=np.mean(pe,axis=0)
    assert np.array_equal(yt,y_test), 'Mismatch with paper test labels/order'
    fn=TARGET_OUT/f'r2_trained_random_mask_seed{seed}.npz'
    np.savez_compressed(fn,test_labels=yt,probs_single=ps,probs_ensemble=pens,seed=seed,expected_retention=RHO)
    pd.DataFrame(hist).to_csv(TARGET_OUT/f'r2_trained_random_mask_seed{seed}_history.csv',index=False)
    return {'Arm':'trained random mask (resampled)','Seed':seed,'AUROC':roc_auc_score(yt,pens),
            'AUPRC':average_precision_score(yt,pens),'Brier':brier_score_loss(yt,pens),
            'Epochs':len(hist),'Expected retention':RHO,'Predictions':str(fn)}


In [ ]:
# RUN ONLY missing seeds; does not delete or overwrite successful earlier runs
rows=[]
if RUN_TRAINED_SPARSE_CONTROL:
    for seed in SPARSE_SEEDS:
        saved=TARGET_OUT/f'r2_trained_random_mask_seed{seed}.npz'
        if saved.exists():
            d=np.load(saved)
            assert np.array_equal(d['test_labels'].astype(int),y_test)
            p=d['probs_ensemble'];print('Using completed seed',seed)
            rows.append({'Arm':'trained random mask (resampled)','Seed':seed,
                         'AUROC':roc_auc_score(y_test,p),'AUPRC':average_precision_score(y_test,p),
                         'Brier':brier_score_loss(y_test,p),'Epochs':np.nan,
                         'Expected retention':RHO,'Predictions':str(saved)})
        else:
            rows.append(train_sparse_control(seed))
    trained=pd.DataFrame(rows)
    assert len(trained)==3
    trained.to_csv(TARGET_OUT/'R2_independently_trained_sparse_baseline.csv',index=False)
    display(trained)
    display(trained[['AUROC','AUPRC','Brier']].agg(['mean','std']))
else: print('Training explicitly disabled; no trained-control evidence produced.')


### Comparison against *existing* matched ON/OFF outputs

Do **not** confuse the above newly trained random-mask model with A6 frozen-model random intervention. Compare paired seeds with already existing `gating_ON_seed*_predictions.npz` and `gating_OFF_seed*_predictions.npz`.

In [ ]:
comp=[]
for seed in SPARSE_SEEDS:
    for arm in ['ON','OFF']:
        f=REV_DIR/f'gating_{arm}_seed{seed}_predictions.npz'
        assert f.exists(), f'Missing previously trained {arm} seed {seed}: {f}'
        d=np.load(f,allow_pickle=True); assert np.array_equal(d['test_labels'].astype(int),y_test)
        p=d['probs_ensemble']
        comp.append({'Arm':'learned gating' if arm=='ON' else 'dense gating OFF','Seed':seed,
                     'AUROC':roc_auc_score(y_test,p),'AUPRC':average_precision_score(y_test,p),
                     'Brier':brier_score_loss(y_test,p)})
    f=TARGET_OUT/f'r2_trained_random_mask_seed{seed}.npz'
    assert f.exists(),f'Sparse training not yet complete for seed {seed}'
    d=np.load(f);p=d['probs_ensemble']
    comp.append({'Arm':'trained random mask (resampled)','Seed':seed,
                 'AUROC':roc_auc_score(y_test,p),'AUPRC':average_precision_score(y_test,p),
                 'Brier':brier_score_loss(y_test,p)})
comparison=pd.DataFrame(comp)
assert comparison.groupby('Arm')['Seed'].nunique().eq(3).all()
comparison.to_csv(TARGET_OUT/'R2_three_arm_matched_seeds.csv',index=False)
display(comparison)
display(comparison.groupby('Arm')[['AUROC','AUPRC','Brier']].agg(['mean','std']))
